# 02 · LangGraph 基础（StateGraph / Node / Edge / conditional_edges）

**对应章节**：LangGraph 教程第 02 章 —— 用 StateGraph 搭图、定义节点与边、条件分支、图可视化。

**演示概念**：
- `StateGraph` + `TypedDict` 状态、`add_node` / `add_edge` / `add_conditional_edges`；
- `START` / `END` 特殊节点；
- 用 `get_graph().draw_png()` 把图导出为图片（需要 graphviz / pygraphviz，见下方注意）。

**运行前置**：
- 需要 `.env`（OPENAI_API_KEY / OPENAI_API_BASE / OPENAI_MODEL），放在仓库根目录（与 .env.example 同位置）；
- 需要已 `uv sync`。
- ⚠️ `promptChain.py` 会真实调用模型，所以**必须**有可用的 API Key；
  `promptChainWithReview.py` 末尾的 `draw_png('./promptchain.png')` 会生成图片到 notebook 当前目录，
  需系统安装 graphviz（`dot` 命令）以及 Python 包 `pygraphviz`（pyproject 已含）。

## 公共头部：引入 LLM 客户端

本 notebook 依赖 `structured`。先把仓库根目录加入 `sys.path`，再从 `src.agent_cookbook` 引入 `structured`。
后续代码格不再重复引入。

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # 仓库根目录，使 src 包可被导入
from src.agent_cookbook import structured

## 1) 最简 StateGraph：`simpleGraph.py`

不依赖 LLM。演示如何用 `StateGraph` 定义两个节点 `node1`/`node2`，并用 `add_edge`
连成 `START -> node1 -> node2` 与 `node1 -> END` 的两条分支。运行后打印图的 nodes / channels 结构。

In [ ]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Literal,Annotated
from pydantic import BaseModel,Field
from pprint import pprint

    

class AgentState(TypedDict):
    content: Annotated[str,lambda old,new: old +new]


def Node1(state: AgentState) -> dict:
    return {'content': 'node1'}

def Node2(state: AgentState) -> dict:
    return {'content': 'node2'}

def main():
    graph = StateGraph(AgentState)
    workflow = graph.add_node("node1",Node1)\
                        .add_node("node2",Node2)\
                        .add_edge(START,"node1")\
                        .add_edge("node1","node2")\
                        .add_edge("node1",END).compile()

    result = workflow.invoke(AgentState(content="begin"))  
    print('####nodes:\n')
    pprint(workflow.nodes)

    print('####channels:\n')
    pprint(workflow.channels)


    for name,node in workflow.nodes.items():
        print(f'\n\n***node: {name}:')
        print(f'{name} subscribe channels: {node.channels}')
        print(f'{name} triggers: {node.triggers}')
        print(f'{name} writers:')
        pprint(node.writers)






# from pprint import pprint
# pprint(workflow.channels)
# pprint(workflow.nodes)
# print(f'###\n')
# pprint(workflow.nodes["genJoke"].channels)
# pprint(workflow.nodes["genJoke"].triggers)
# print(f'###\n')
# pprint(workflow.nodes["humanReview"].channels)
# pprint(workflow.nodes["humanReview"].triggers)

if __name__ == "__main__":
    main()

## 2) 带模型的链式图：`promptChain.py`

依赖 `structured`。演示：“生成笑话 → 翻译” 的线性两节点链，输入 `topic` 输出 `content`。
运行会真实调用模型（需要 API Key）。

In [ ]:
from typing import TypedDict
from langgraph.graph import START,StateGraph,END
from pydantic import BaseModel,ValidationError
from dotenv import load_dotenv

load_dotenv()

class Joke(BaseModel):
    joke: str


class AgentState(TypedDict):
    topic: str
    content: str


def genJoke(state: AgentState):
    
    response = structured(Joke, [{"role":"user","content":f"gen a Joek about {state['topic']}"}])
    #response.pretty_print()
    print(f'\n content: {response.joke}\n')
    return {"content": response.joke}

def translate(state: AgentState):

    response = structured(Joke, [{"role":"user","content":f"translate to chinese: {state['content']}"}])

    return {"content": response.joke}

graph = StateGraph(AgentState).add_node("genJoke",genJoke)\
                              .add_node("translate",translate)\
                              .add_edge(START,"genJoke")\
                              .add_edge("genJoke","translate")\
                              .add_edge("translate",END)
workflow = graph.compile()
#print(f'#### graph : \n {workflow.get_graph().draw_ascii()}\n')
result = workflow.invoke({"topic":"wednesday","content":""},{"configurable": {"thread_id": "foo"}})
print(f'#### result: \n{result}')

## 3) 条件边的笑话工坊：`promptChainWithReview.py`

依赖 `structured`。在链中间插入一个“评审”节点 `reviewJoke`，再由 `checkReviewResult`
做 `add_conditional_edges` 分支：评审通过去翻译，否则回到生成节点重试（带最大重试次数）。

> 注意：`workflow.get_graph().draw_png('./promptchain.png')` 会把图渲染成 `promptchain.png`
> 到 notebook 当前目录（examples/）。若缺少 graphviz / pygraphviz 会报错，可注释掉该行。

In [ ]:
from typing import TypedDict, Optional, Literal
from langgraph.graph import START, StateGraph, END
from langgraph.config import get_config
from pydantic import BaseModel, Field
from dotenv import load_dotenv
load_dotenv()


class Joke(BaseModel):
    joke: str


class CriticResult(BaseModel):
    isFunny: bool = Field(description="is joke funny or not")
    opinion: str = Field(description="how to improve")


class AgentState(TypedDict):
    topic: str
    content: str
    reviewResult: Optional[CriticResult]
    retryCount: int


def genJoke(state: AgentState):
    message: str = f'gen a Joke about {state["topic"]}'
    if state["reviewResult"]:
        message = message + f', consider the opinion: {state["reviewResult"].opinion}'

    print(f'# gen \n gen message: {message}')
    response = structured(Joke, [{"role":"user","content":message}])
    return {"content": response.joke}


def reviewJoke(state: AgentState):
    response = structured(CriticResult, [
            {"role":"system","content":"You are a strict joke reviewer."},
            {"role":"user","content":f'is this Joke funny or not ? Joke: {state["content"]},\
             if not let me know how to improve it'}
        ])
    print(f'## review: \n is funny: {response.isFunny}, opinion: {response.opinion}')
    return {"reviewResult": response, "retryCount": state["retryCount"] + 1}


def checkReviewResult(state: AgentState) -> Literal["genJoke", "translate"]:
    config = get_config()
    max_retries = config.get("configurable", {}).get("max_retries", 3)

    if state["reviewResult"] and state["reviewResult"].isFunny:
        print("✅ 评审通过")
        return "translate"

    if state["retryCount"] >= max_retries:
        print(f"⚠️ 达到最大重试次数 {max_retries}，强制继续")
        return "translate"

    print(f"❌ 评审未通过，重试第 {state['retryCount'] + 1}/{max_retries} 次")
    return "genJoke"


def translate(state: AgentState):
    response = structured(Joke, [{"role":"user","content":f"translate to chinese: {state['content']}"}])
    return {"content": response.joke}


graph = StateGraph(AgentState).add_node("genJoke", genJoke)\
                              .add_node("review", reviewJoke)\
                              .add_node("translate", translate)\
                              .add_edge(START, "genJoke")\
                              .add_edge("genJoke", "review")\
                              .add_conditional_edges(
                                  "review", checkReviewResult,
                                  {"genJoke": "genJoke", "translate": "translate"})\
                              .add_edge("translate", END)

workflow = graph.compile()
# result = workflow.invoke(
#     {"topic":"wednesday","content":"","reviewResult":None,"retryCount":0},
#     {"configurable": {"thread_id": "foo", "max_retries": 2}})
# print(f'#### result: \n{result}')

workflow.get_graph().print_ascii()
workflow.get_graph().draw_png('./promptchain.png')

### 小结
- `conditional_edges` 的分支函数返回“目标节点名”或 `Literal` 取值，决定下一步走向；
- 状态用 `Annotated[str, lambda old,new: old+new]` 可以做累加 / 合并；
- 图结构可视化是调试 Agent 流程的利器，建议养成 `draw_*` 的习惯。